In [507]:
# Start with the CFS (2017)
import pyxlsb
import pandas as pd
cfs_path = '/Users/jeffreyohl/Dropbox/SpatialW25/BigDataFiles/pset2/CFS/'
cfs_mapping_path = "/Users/jeffreyohl/Dropbox/SpatialW25/BigDataFiles/pset2/Mapping Files/CFS_sector_aggregation.xlsx"
wiod_path = "/Users/jeffreyohl/Dropbox/SpatialW25/BigDataFiles/pset2/WIOD/WIOT2014_Nov16_ROW.xlsb"
wiod_mapping_sector_path = "/Users/jeffreyohl/Dropbox/SpatialW25/BigDataFiles/pset2/Mapping Files/WIOD_sector_aggregation.xlsx"
out_folder = "/Users/jeffreyohl/Dropbox/SpatialW25/BigDataFiles/pset2/X_in_j/"
out_folder_IO = '/Users/jeffreyohl/Dropbox/SpatialW25/BigDataFiles/pset2/IO/'
out_folder_gamma = '/Users/jeffreyohl/Dropbox/SpatialW25/BigDataFiles/pset2/gamma/'

out_folder_GO = '/Users/jeffreyohl/Dropbox/SpatialW25/BigDataFiles/pset2/GO/'
countries_path = "/Users/jeffreyohl/Dropbox/SpatialW25/BigDataFiles/pset2/Mapping Files/WIOD_country_codes.csv"

intial_emp = "/Users/jeffreyohl/Dropbox/SpatialW25/BigDataFiles/pset2/dummy data/L0_initial.csv"

In [350]:
country = pd.read_csv(countries_path)

In [351]:
country.head()

,Code,CountryLong,Country
0,AUS,Australia,1
1,AUT,Austria,2
2,BEL,Belgium,3
3,BGR,Bulgaria,4
4,BRA,Brazil,5


In [352]:
country_map = country.set_index('Code')['Country'].to_dict()


# WIOD 2014

In [353]:


# ------------------------------
# 1) Read the matrix from Excel
# ------------------------------
# - skiprows=4 so that the first row we read is row 5 of the sheet
# - header=[0,1] means: the first two rows we read become a 2-level column header
#   (row 5 is level 0: "ImportingCountry", row 6 is level 1: "ImportingIndustry")
# - index_col=[2,3] means: use columns C and D (zero-based: 2 and 3) as the row index
#   (ExportingCountry, ExportingIndustry)
df = pd.read_excel(
    wiod_path,
    sheet_name="2014",
    skiprows=4,        # Adjust if you have more or fewer label rows
    header=[0, 1],     # Row 5 => upper col labels, Row 6 => lower col labels
    index_col=[2, 3],  # Column C => ExportingCountry, Column D => ExportingIndustry
    # usecols="C:ZZ",  # Optionally restrict columns if needed
)

# By default, pandas will name the two column levels something like (None, None).
# We can rename them:
df.columns.names = ["ImportingCountry", "ImportingIndustry"]
df.index.names = ["ExportingCountry", "ExportingIndustry"]




# df is now a 2D table whose:
#   - row index = (ExportingCountry, ExportingIndustry)
#   - columns   = (ImportingCountry, ImportingIndustry)
#   - cell values = trade flows (or I/O flows).

# ------------------------------
# 2) Reshape from wide to long
# ------------------------------
# We "stack" over the two column levels:
df_long = df.stack(level=[0, 1]).reset_index()

# The result has columns:
#   ["ExportingCountry", "ExportingIndustry", "ImportingCountry", "ImportingIndustry", 0]
# Rename the last one to "Value" (or something relevant).
df_long.columns = [
    "ExportingCountry",
    "ExportingIndustry",
    "ImportingCountry",
    "ImportingIndustry",
    "Value",
]

# ------------------------------
# 3) Filter out invalid industries (optional)
# ------------------------------
# If valid industries are r1..r56 for exporting and c1..c56 for importing:
valid_r = [f"r{i}" for i in range(1, 57)]
valid_c = [f"c{i}" for i in range(1, 57)]

df_long = df_long[
    df_long["ExportingIndustry"].isin(valid_r)
    & df_long["ImportingIndustry"].isin(valid_c)
]

# ------------------------------
# 4) Collapse certain countries
# ------------------------------
# E.g. CHE, HRV, LUX, LVA, MLT, NOR => "ROW"
row_countries = ["CHE", "HRV", "LUX", "LVA", "MLT", "NOR"]

df_long["ExportingCountry"] = df_long["ExportingCountry"].where(
    ~df_long["ExportingCountry"].isin(row_countries), 
    "ROW"
)
df_long["ImportingCountry"] = df_long["ImportingCountry"].where(
    ~df_long["ImportingCountry"].isin(row_countries), 
    "ROW"
)

# ------------------------------
# 5) Summation / Aggregation
# ------------------------------
df_agg = df_long.groupby(
    ["ExportingCountry", "ExportingIndustry", 
     "ImportingCountry", "ImportingIndustry"],
    as_index=False
)["Value"].sum()


print(df_agg.head(15))


/var/folders/g9/ggrmf_5j6tx4rv5qdhs0slq40000gn/T/ipykernel_45649/3192670138.py:35: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df_long = df.stack(level=[0, 1]).reset_index()


   ExportingCountry ExportingIndustry ImportingCountry ImportingIndustry  \
0               AUS                r1              AUS                c1   
1               AUS                r1              AUS               c10   
2               AUS                r1              AUS               c11   
3               AUS                r1              AUS               c12   
4               AUS                r1              AUS               c13   
5               AUS                r1              AUS               c14   
6               AUS                r1              AUS               c15   
7               AUS                r1              AUS               c16   
8               AUS                r1              AUS               c17   
9               AUS                r1              AUS               c18   
10              AUS                r1              AUS               c19   
11              AUS                r1              AUS                c2   
12          

In [354]:
df_agg.columns

Index(['ExportingCountry', 'ExportingIndustry', 'ImportingCountry',
       'ImportingIndustry', 'Value'],
      dtype='object')

In [355]:
df_agg

,ExportingCountry,ExportingIndustry,ImportingCountry,ImportingIndustry,Value
0,AUS,r1,AUS,c1,12924.179691
1,AUS,r1,AUS,c10,0.226825
2,AUS,r1,AUS,c11,86.721589
3,AUS,r1,AUS,c12,109.147573
4,AUS,r1,AUS,c13,147.252719
...,...,...,...,...,...
4528379,USA,r9,USA,c56,0.0
4528380,USA,r9,USA,c6,743.552862
4528381,USA,r9,USA,c7,75.372114
4528382,USA,r9,USA,c8,208.748822


In [356]:
country_map

{'AUS': 1,
 'AUT': 2,
 'BEL': 3,
 'BGR': 4,
 'BRA': 5,
 'CAN': 6,
 'CHN': 7,
 'CYP': 8,
 'CZE': 9,
 'DNK': 10,
 'EST': 11,
 'FIN': 12,
 'FRA': 13,
 'DEU': 14,
 'GRC': 15,
 'HUN': 16,
 'IND': 17,
 'IDN': 18,
 'ITA': 19,
 'IRL': 20,
 'JPN': 21,
 'LTU': 22,
 'MEX': 23,
 'NLD': 24,
 'POL': 25,
 'PRT': 26,
 'ROU': 27,
 'RUS': 28,
 'ESP': 29,
 'SVK': 30,
 'SVN': 31,
 'KOR': 32,
 'SWE': 33,
 'TWN': 34,
 'TUR': 35,
 'GBR': 36,
 'ROW': 37,
 'USA': 38}

In [357]:
country_map

{'AUS': 1,
 'AUT': 2,
 'BEL': 3,
 'BGR': 4,
 'BRA': 5,
 'CAN': 6,
 'CHN': 7,
 'CYP': 8,
 'CZE': 9,
 'DNK': 10,
 'EST': 11,
 'FIN': 12,
 'FRA': 13,
 'DEU': 14,
 'GRC': 15,
 'HUN': 16,
 'IND': 17,
 'IDN': 18,
 'ITA': 19,
 'IRL': 20,
 'JPN': 21,
 'LTU': 22,
 'MEX': 23,
 'NLD': 24,
 'POL': 25,
 'PRT': 26,
 'ROU': 27,
 'RUS': 28,
 'ESP': 29,
 'SVK': 30,
 'SVN': 31,
 'KOR': 32,
 'SWE': 33,
 'TWN': 34,
 'TUR': 35,
 'GBR': 36,
 'ROW': 37,
 'USA': 38}

In [358]:
df_agg['ExportingCountry'] = df_agg['ExportingCountry'].map(country_map)
df_agg['ImportingCountry'] = df_agg['ImportingCountry'].map(country_map)

In [359]:
df_agg = df_agg.to_csv(out_folder + "long_fix_countries.csv")

In [360]:
df_agg = pd.read_csv(out_folder + "long_fix_countries.csv")

In [361]:
df_agg

,Unnamed: 0,ExportingCountry,ExportingIndustry,ImportingCountry,ImportingIndustry,Value
0,0,1,r1,1,c1,12924.179691
1,1,1,r1,1,c10,0.226825
2,2,1,r1,1,c11,86.721589
3,3,1,r1,1,c12,109.147573
4,4,1,r1,1,c13,147.252719
...,...,...,...,...,...,...
4528379,4528379,38,r9,38,c56,0.000000
4528380,4528380,38,r9,38,c6,743.552862
4528381,4528381,38,r9,38,c7,75.372114
4528382,4528382,38,r9,38,c8,208.748822


In [362]:
mapping = pd.read_excel(wiod_mapping_sector_path)

In [363]:
%pip install openpyxl
import openpyxl

Note: you may need to restart the kernel to use updated packages.


In [364]:
# 1) Build lookup dictionaries from your ‘mapping’ dataframe
out_map = mapping.set_index('output_code')['OrderExcldNon'].to_dict()
in_map  = mapping.set_index('input_code')['OrderExcldNon'].to_dict()
df_agg_sector_fix = df_agg.copy()
# 2) Map the old codes in df_agg to the new “Order in CDP 2019”
df_agg_sector_fix['ExportingIndustry'] = df_agg_sector_fix['ExportingIndustry'].map(out_map)
df_agg_sector_fix['ImportingIndustry'] = df_agg_sector_fix['ImportingIndustry'].map(in_map)

# 3) Drop rows where either mapped industry is NaN
df_agg_sector_fix = df_agg_sector_fix.dropna(subset=['ExportingIndustry','ImportingIndustry'])

# 4) Collapse (aggregate) non-unique rows, for example by summing 'Value'
df_agg_sector_fix = df_agg_sector_fix.groupby(
    ['ExportingCountry','ExportingIndustry','ImportingCountry','ImportingIndustry'],
    as_index=False
)['Value'].sum()


In [365]:
# For example, save to CSV
df_agg_sector_fix.to_csv(out_folder+ "long_fix_countries_and_sectors.csv", index=False)


In [366]:
# For example, save to CSV
df_agg_sector_fix =pd.read_csv(out_folder+ "long_fix_countries_and_sectors.csv")


In [367]:
df_agg_sector_fix

,ExportingCountry,ExportingIndustry,ImportingCountry,ImportingIndustry,Value
0,1,1.0,1,1.0,10851.759013
1,1,1.0,1,2.0,262.228559
2,1,1.0,1,3.0,40.510798
3,1,1.0,1,4.0,2.897519
4,1,1.0,1,5.0,352.067899
...,...,...,...,...,...
698891,38,22.0,38,18.0,214761.185588
698892,38,22.0,38,19.0,25490.798838
698893,38,22.0,38,20.0,241208.476296
698894,38,22.0,38,21.0,116928.130124


In [368]:
us_code = country_map['USA']

In [369]:
us_code

38

In [370]:
domestic_usa_trade = df_agg_sector_fix[
    (df_agg_sector_fix['ExportingCountry'] == us_code) & 
    (df_agg_sector_fix['ImportingCountry'] == us_code )
]

# Group by exporting industry and sum Value
domestic_expenditure = domestic_usa_trade.groupby('ExportingIndustry', as_index=False)['Value'].sum()


In [371]:
domestic_expenditure

,ExportingIndustry,Value
0,1.0,3.702758e+05
1,2.0,5.021600e+04
2,3.0,2.875896e+05
3,4.0,3.924727e+05
4,5.0,3.776814e+05
5,6.0,1.617610e+05
6,7.0,9.119060e+04
7,8.0,5.297387e+05
8,9.0,1.214166e+05
9,10.0,2.010416e+05


In [372]:
import pandas as pd

# Replace 'your_file.dta' with the path to your .dta file
cfs_2017 = pd.read_stata(cfs_path + 'states_ind_bilateral_trade2017.dta')


In [373]:
cfs_2017['origin'] = cfs_2017['origin'].replace('District of Columbia', 'Virginia')
cfs_2017['dest'] = cfs_2017['dest'].replace('District of Columbia', 'Virginia')

In [374]:
cfs_2017['origin'].unique()

array(['Arkansas', 'South Carolina', 'Maryland', 'Delaware', 'Wisconsin',
       'Massachusetts', 'Iowa', 'Colorado', 'Montana', 'Louisiana',
       'Kentucky', 'Idaho', 'New Mexico', 'Florida', 'Washington',
       'Hawaii', 'Texas', 'Missouri', 'California', 'Alaska', 'Utah',
       'Arizona', 'Nevada', 'Wyoming', 'Illinois', 'North Dakota',
       'Indiana', 'Georgia', 'Connecticut', 'New York', 'Mississippi',
       'North Carolina', 'Maine', 'Oklahoma', 'Alabama', 'Michigan',
       'Virginia', 'Nebraska', 'South Dakota', 'New Hampshire', 'Vermont',
       'Pennsylvania', 'West Virginia', 'Ohio', 'Kansas', 'New Jersey',
       'Minnesota', 'Oregon', 'Tennessee', 'Rhode Island',
       'United States'], dtype=object)

In [375]:
cfs_2017 = cfs_2017[cfs_2017['origin_abrv'] != 'USA']


cfs_2017 = cfs_2017[cfs_2017['dest_abrv'] != 'USA']

# Display the first few rows of the dataframe
cfs_2017.head()

cfs_2017.to_csv(cfs_path + 'states_ind_bilateral_trade2017.csv')

In [376]:
cfs_2017

,origin,origin_abrv,dest,dest_abrv,year,naics2012,naics2012_ttl,naics2012_f,val,ton,tmile,avgmile,d_state_origin,d_state_dest
0,Arkansas,AR,Alabama,AL,2017,423,"Merchant wholesalers, durable goods",,369,252,104,405,1,1
1,South Carolina,SC,Alabama,AL,2017,333,Machinery manufacturing,,83,4,2,618,1,1
2,Maryland,MD,Alabama,AL,2017,423,"Merchant wholesalers, durable goods",,110,0,0,830,1,1
3,Delaware,DE,Alabama,AL,2017,4239,Miscellaneous durable goods merchant wholesalers,,0,0,0,848,1,1
4,Wisconsin,WI,Alabama,AL,2017,4236,Household appliances and electrical and electr...,,0,0,0,798,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101505,Colorado,CO,Wyoming,WY,2017,4238,"Machinery, equipment, and supplies merchant wh...",,156,0,2,277,1,1
101506,Missouri,MO,Wyoming,WY,2017,327,Nonmetallic mineral product manufacturing,,0,0,0,0,1,1
101507,Arkansas,AR,Wyoming,WY,2017,31-33,Manufacturing,,86,0,0,1224,1,1
101508,Maryland,MD,Wyoming,WY,2017,42,Wholesale trade,,0,0,0,1826,1,1


In [377]:
cfs_2017 = cfs_2017[['dest', 'origin', 'naics2012', 'val']]

# some summary stats

In [378]:
# Group by naics2012, sum the val column, and sort in descending order
cfs_2017 = cfs_2017[cfs_2017['naics2012'] != '00']
result = cfs_2017.groupby('naics2012')['val'].sum().sort_values(ascending=False)


In [379]:
cfs_mapping = pd.read_excel(cfs_mapping_path)


In [380]:
cfs_mapping['naics2012'] = cfs_mapping['naics2012'].astype(str)
cfs_2017['naics2012'] = cfs_2017['naics2012'].astype(str)

# Create a mapping dictionary from cfs_mapping
mapping_dict = dict(zip(cfs_mapping['naics2012'], cfs_mapping['CDP order']))

# Map the 'naics_2012' column in cfs_2017 to a new 'CDP order' column
cfs_2017['CDP order'] = cfs_2017['naics2012'].map(mapping_dict)


In [381]:
cfs_2017_collapsed = cfs_2017.groupby(['dest', 'origin', 'CDP order'], as_index=False)['val'].sum()

In [382]:
cfs_2017['naics2012'].unique()

array(['423', '333', '4239', '4236', '337', '4237', '324', '339', '4249',
       '325', '4541', '42', '326', '424', '321', '4241', '335', '4243',
       '312', '4233', '4234', '316', '313', '334', '4232', '4244', '331',
       '336', '4245', '4931', '4231', '31-33', '5111', '327', '332',
       '315', '4242', '323', '4238', '4247', '551114', '4235', '212',
       '311', '4248', '322', '45431', '4246', '314'], dtype=object)

In [383]:
cfs_2017_collapsed

,dest,origin,CDP order,val
0,Alabama,Alabama,1.0,4427
1,Alabama,Alabama,2.0,189
2,Alabama,Alabama,3.0,3160
3,Alabama,Alabama,4.0,4306
4,Alabama,Alabama,5.0,2269
...,...,...,...,...
28965,Wyoming,Wyoming,9.0,72
28966,Wyoming,Wyoming,10.0,4
28967,Wyoming,Wyoming,11.0,0
28968,Wyoming,Wyoming,12.0,4406


In [384]:
import pandas as pd

# Suppose your DataFrame is called df and has columns:
#   ['dest_abrv', 'origin_abrv', 'CDP order', 'val']
# Make sure df is read in as a DataFrame.

# Get the sorted list of all states
all_states = sorted(set(cfs_2017_collapsed['dest']) | set(cfs_2017_collapsed['origin']))

# Get all unique sectors
cdp_orders = sorted(cfs_2017_collapsed['CDP order'].unique())

# Dictionary to store the resulting 50x50 share matrices per sector
trade_share_matrices = {}

for sector in cdp_orders:
    # Filter to this sector
    sector_df = cfs_2017_collapsed[cfs_2017_collapsed['CDP order'] == sector].copy()

    # Pivot so that rows=origin, columns=destination, values=val
    pivoted = sector_df.pivot_table(
        index='origin',
        columns='dest',
        values='val',
        aggfunc='sum',
        fill_value=0
    )

    # Reindex to ensure we have all 50 states as rows & columns (fill missing with 0)
    pivoted = pivoted.reindex(index=all_states, columns=all_states, fill_value=0)

    # Column sums (each destination state's total expenditure in this sector)
    col_sums = pivoted.sum(axis=0)

    # Divide each column by its sum to get shares; fill any 0/0 with 0
    shares = pivoted.div(col_sums, axis=1).fillna(0)

    # Store in a dictionary keyed by the sector (CDP order)
    trade_share_matrices[sector] = shares

# Now trade_share_matrices[sector] is your 50x50 DataFrame of shares 
# for that particular sector. Each column sums to 1.


In [385]:
trade_share_matrices

{np.float64(1.0): dest             Alabama    Alaska   Arizona  Arkansas  California  Colorado  \
 origin                                                                         
 Alabama         0.379089  0.000000  0.007258  0.014605    0.003261  0.008314   
 Alaska          0.000000  0.948381  0.000388  0.000000    0.000396  0.000000   
 Arizona         0.000000  0.000000  0.472241  0.000648    0.014096  0.006176   
 Arkansas        0.000000  0.000000  0.015791  0.447114    0.007238  0.015045   
 California      0.035194  0.004223  0.195700  0.020200    0.666872  0.073719   
 Colorado        0.001456  0.000000  0.006538  0.000000    0.014637  0.526249   
 Connecticut     0.000514  0.000000  0.000000  0.000000    0.000183  0.000079   
 Delaware        0.000514  0.000000  0.000111  0.000000    0.000914  0.000000   
 Florida         0.014129  0.000000  0.001884  0.003180    0.003391  0.001504   
 Georgia         0.121596  0.000000  0.002826  0.027032    0.006857  0.007918   
 Hawaii    

In [386]:
df_shares

,CDP order,dest,val,sector_total,share
0,1.0,Alabama,11678,1072121,0.010892
1,1.0,Alaska,2131,1072121,0.001988
2,1.0,Arizona,18048,1072121,0.016834
3,1.0,Arkansas,16980,1072121,0.015838
4,1.0,California,131247,1072121,0.122418
...,...,...,...,...,...
645,13.0,Virginia,367153,19524993,0.018804
646,13.0,Washington,438802,19524993,0.022474
647,13.0,West Virginia,84954,19524993,0.004351
648,13.0,Wisconsin,371530,19524993,0.019028


In [387]:
domestic_expenditure

,ExportingIndustry,Value
0,1.0,3.702758e+05
1,2.0,5.021600e+04
2,3.0,2.875896e+05
3,4.0,3.924727e+05
4,5.0,3.776814e+05
5,6.0,1.617610e+05
6,7.0,9.119060e+04
7,8.0,5.297387e+05
8,9.0,1.214166e+05
9,10.0,2.010416e+05


# Get grand expenditure share (CFS)

In [388]:
df_sum_by_dest = cfs_2017_collapsed.groupby('dest')['val'].sum()

# 2. Compute the grand total of 'val'.
grand_total = cfs_2017_collapsed['val'].sum()

# 3. Divide each state's total by the grand total to get the share.
grand_expenditure_shares = df_sum_by_dest / grand_total



In [389]:
grand_expenditure_shares.sort_values(ascending=False)

dest
Texas             0.132063
California        0.113394
Illinois          0.051990
New York          0.050691
Florida           0.048189
Ohio              0.040973
Pennsylvania      0.036634
Michigan          0.035739
Georgia           0.033333
New Jersey        0.028266
Indiana           0.027043
North Carolina    0.025998
Washington        0.024571
Tennessee         0.023789
Louisiana         0.020626
Wisconsin         0.020561
Virginia          0.020080
Minnesota         0.018228
Massachusetts     0.017571
Missouri          0.016816
Kentucky          0.015911
Alabama           0.014816
South Carolina    0.014371
Arizona           0.013864
Colorado          0.012609
Maryland          0.011918
Iowa              0.011628
Kansas            0.011547
Oklahoma          0.010791
Connecticut       0.010598
Oregon            0.010468
Mississippi       0.008752
Utah              0.007802
Arkansas          0.007513
Nebraska          0.006926
Nevada            0.005758
New Hampshire     0.003

In [390]:
grand_expenditure_shares.sum()

np.float64(1.0)

# Get each state's share of [that sector's expenditure]

In [391]:
cfs_2017_collapsed.columns

Index(['dest', 'origin', 'CDP order', 'val'], dtype='object')

In [392]:
import pandas as pd

# Suppose your DataFrame is called df and has columns:
#   ['dest_abbrv', 'origin_abbrv', 'CDP order', 'val']

# 1) Sum over all origins to get total spending by (CDP order, dest_abbrv).
df_sum = cfs_2017_collapsed.groupby(['CDP order', 'dest'], as_index=False)['val'].sum()

# 2) Compute total expenditures by each sector (CDP order).
df_total = df_sum.groupby('CDP order', as_index=False)['val'].sum()
df_total = df_total.rename(columns={'val': 'sector_total'})

# 3) Merge back to compute shares
df_shares = pd.merge(df_sum, df_total, on='CDP order', how='left')
df_shares['share'] = df_shares['val'] / df_shares['sector_total']

# df_shares now has columns:
#   ['CDP order', 'dest_abbrv', 'val', 'sector_total', 'share']
# where share is the fraction of sector expenditure going to each state.


In [393]:
df_shares

,CDP order,dest,val,sector_total,share
0,1.0,Alabama,11678,1072121,0.010892
1,1.0,Alaska,2131,1072121,0.001988
2,1.0,Arizona,18048,1072121,0.016834
3,1.0,Arkansas,16980,1072121,0.015838
4,1.0,California,131247,1072121,0.122418
...,...,...,...,...,...
645,13.0,Virginia,367153,19524993,0.018804
646,13.0,Washington,438802,19524993,0.022474
647,13.0,West Virginia,84954,19524993,0.004351
648,13.0,Wisconsin,371530,19524993,0.019028


In [394]:
import pandas as pd

# trade_share_matrices[sector] = 50×50 DataFrame of shares
# domestic_expenditure has columns ['ExportingIndustry','Value']
# df_shares has columns ['CDP order','dest_abrv','shares']

scaled_trade_flows = {}  # to store the final dollar‐value trade flow matrices

# Get the list of unique sectors from the trade_share_matrices keys
all_sectors = list(trade_share_matrices.keys())

for sector in all_sectors:
    # 1. Retrieve the 50×50 “trade shares” matrix for this sector
    share_matrix = trade_share_matrices[sector].copy()
    
    # 2. Find the total U.S. expenditure for this sector
    #    (matching sector to ExportingIndustry in domestic_expenditure)
    sector_value = domestic_expenditure.loc[
        domestic_expenditure['ExportingIndustry'] == sector,
        'Value'
    ].values[0]
    
    # 3. Get the share of that total going to each destination state
    #    from df_shares (matching sector to CDP order)
    sector_shares_df = df_shares[df_shares['CDP order'] == sector]
    # Make it a Series indexed by destination state
    sector_shares_by_state = sector_shares_df.set_index('dest')['share']
    
    # Ensure it matches the columns in the share_matrix (50 states)
    sector_shares_by_state = sector_shares_by_state.reindex(
        share_matrix.columns,
        fill_value=0
    )
    
    # 4. Create the column‐by‐column scaling factor:
    #    "sector_value * sector_shares_by_state"
    #    (this is a Series whose index = states, same as share_matrix.columns)
    scale_by_state = sector_shares_by_state * sector_value
    
    # 5. Multiply each column of the 50×50 matrix by the column’s scale factor
    scaled_matrix = share_matrix.mul(scale_by_state, axis=1)
    
    # 6. Store the resulting (dollar‐valued) 50×50 matrix in a dictionary
    scaled_trade_flows[sector] = np.array(scaled_matrix)

# After this loop, scaled_trade_flows[sector] is the 50×50 matrix
# of trade flows in dollars for that sector, where each column has been
# scaled by [sector_value * share for that (sector,dest_state)].


# scaled_trade_flows is between-state, sectors 1-13

# Check totals.

In [395]:
sums_dict = {}
for key, matrix in scaled_trade_flows.items():
        # Calculate the sum of all elements in the matrix
        matrix_sum = matrix.sum()
        sums_dict[key] = matrix_sum
    


In [396]:
sums_dict

{np.float64(1.0): np.float64(370275.8303016197),
 np.float64(2.0): np.float64(50215.998282914414),
 np.float64(3.0): np.float64(287589.56145522813),
 np.float64(4.0): np.float64(392472.7117530798),
 np.float64(5.0): np.float64(377681.35514771746),
 np.float64(6.0): np.float64(161761.02304897114),
 np.float64(7.0): np.float64(91190.6043977333),
 np.float64(8.0): np.float64(529738.6981184247),
 np.float64(9.0): np.float64(121416.63225651279),
 np.float64(10.0): np.float64(201041.62447004678),
 np.float64(11.0): np.float64(277964.2173746787),
 np.float64(12.0): np.float64(83131.51020187109),
 np.float64(13.0): np.float64(880574.2975308779)}

# Bilateral expenditure shares 
# across US states [no sectors?]

In [397]:
import pandas as pd
import numpy as np

# Example: assume your DataFrame is called cfs_2017_collapsed
df = cfs_2017_collapsed

# Step 1: (Optional) group if you have multiple rows per (origin, destination).
#         If each pair is already unique, you can skip this groupby.
df_agg = df.groupby(['origin', 'dest'], as_index=False)['val'].sum()

# Step 2: pivot so that rows=origins, columns=destinations
pivot_df = df_agg.pivot_table(
    index='origin',
    columns='dest',
    values='val',
    aggfunc='sum',      # 'sum' if not already unique
    fill_value=0
)

# Step 3: create shares so that each column sums to 1
# (i.e., share of each destination's total spending on each origin)
pivot_shares = pivot_df.div(pivot_df.sum(axis=0), axis=1)

# Step 4: convert to a 2D NumPy array
result_matrix = pivot_shares.to_numpy()

print(result_matrix)


[[4.85346909e-01 5.87613116e-05 4.21697929e-03 ... 3.50070098e-03
  4.19770282e-03 9.05739863e-04]
 [0.00000000e+00 6.18893720e-01 1.66584961e-05 ... 0.00000000e+00
  3.20925292e-06 0.00000000e+00]
 [2.47855482e-03 7.63897050e-04 4.86375730e-01 ... 9.23446302e-05
  2.12452543e-03 5.28699315e-03]
 ...
 [3.60760001e-04 1.95871039e-05 3.52208203e-04 ... 3.79704329e-01
  7.52569809e-04 2.94892048e-04]
 [7.00141632e-03 1.44944568e-03 7.36067548e-03 ... 6.49770398e-03
  4.44407716e-01 3.60189573e-03]
 [1.78153087e-05 1.95871039e-05 3.47448632e-04 ... 8.39496638e-06
  1.42811755e-04 4.39957873e-01]]


In [398]:
result_matrix
column_sums = result_matrix.sum(axis=0)
print(column_sums)

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1.]


In [399]:
pivot_df

dest,Alabama,Alaska,Arizona,Arkansas,California,Colorado,Connecticut,Delaware,Florida,Georgia,...,South Dakota,Tennessee,Texas,Utah,Vermont,Virginia,Washington,West Virginia,Wisconsin,Wyoming
origin,,,,,,,,,,,,,,,,,,,,,
Alabama,217946,3,1772,1903,14537,1136,378,451,31855,44899,...,95,28013,22250,764,19,3011,1332,417,2616,43
Alaska,0,31597,7,0,128,0,0,0,10,0,...,0,0,40,0,2,4,3685,0,2,0
Arizona,1113,39,204378,209,52855,2963,1337,33,5338,1222,...,82,1840,21866,2375,7,2085,3612,11,1324,251
Arkansas,2350,1,1544,99863,6723,806,205,31,2887,4581,...,94,8759,20410,977,1,845,972,68,1644,96
California,19160,1670,80930,5794,2498320,27289,12412,1864,86820,37250,...,1221,28676,160856,34744,788,27062,71057,1880,18527,1534
Colorado,793,89,3344,254,22062,224871,476,39,4250,1814,...,1762,1768,14396,12120,0,1108,2993,51,1729,9799
Connecticut,714,39,1250,138,10151,894,164939,195,6907,2996,...,23,11996,7655,610,2032,2434,1035,104,1685,18
Delaware,104,0,1809,95,2846,418,699,31710,462,655,...,6,1247,1791,73,52,1095,341,17,417,5
Florida,15227,25,3776,1537,24070,2091,3495,681,824388,37943,...,201,8685,26301,772,168,10344,6020,537,4024,126


# CDP 2019 do not say what they do for services. 
# [tbh, might make sense to use the local employment thing here, too. may see if that changes results]

we will impute service trade using the following (knowing we need to match domestic aggregates)

note, all of this will only apply to ExportingIndustry [13,...,22]  in `domestic_expenditure` dataframe which has columns ExportingIndustry and Value (value is total domestic expenditure in that industry)

 (1) First we have total domestic expenditure shares by state. (what percent of domestic expenditure is spent by each state, using CFS.). this is called grand_expenditure_shares, which is a list, with the states in order, as the labels.

 (2) Then we allocate domestic service expenditure from WIOD, called, domestic_expenditure, this is just the 22 sectors dataframe from before

 (3) Then we assume the same aggregate trade shares for goods also applies to services, this is called [result_matrix],2d nparray columns of which are destinatino, rows of which are origin, again states are in same order. Each column sums to 1

 We use these to create a matrix for each sector. 

We want the ratio of the row values for each column to match that row in result_matrix (which are shares), basically. We want the sum total of the matrix to be equal to the sectoral total expenditure from domestic_expenditure. We want the sum of the column to be (grand_expenditure_share) * (domestic_expenditure for that sector)


 # So at the end of this, should have a 50 x 50 matrix for each of the 10 sectors under consideration
 
 (total expenditure in that sector from WIOD, scales the entire sector's matrix)
 (agg. [goods] expenditure share scales each column)
 (agg. [goods] trade shares give each column's dispersion )


In [400]:
first_missing_sector = 14 # I think we have 13 (retail trade)

In [401]:
import numpy as np
import pandas as pd

# Assume these are provided:
# domestic_expenditure: DataFrame with columns ['ExportingIndustry', 'Value']
# result_matrix: 50 x 50 numpy array (each column sums to 1)
# grand_expenditure_shares: numpy array of length 50 (sums to 1)

# Dictionary to hold the imputed matrices for each sector
sector_matrices = {}

# Filter the dataframe for the industries of interest (13 to 22)
sectors = domestic_expenditure[domestic_expenditure['ExportingIndustry'].isin(range(first_missing_sector, 23))]

for _, row in sectors.iterrows():
    sector = row['ExportingIndustry']
    sector_total = row['Value']
    
    # Initialize an empty 50x50 matrix for this sector
    imputed_matrix = np.zeros_like(result_matrix)
    
    # For each destination state (column index j)
    for j in range(result_matrix.shape[1]):
        # Compute allocated total for destination state j
        allocated_total = grand_expenditure_shares[j] * sector_total
        
        # Scale the goods trade shares for column j
        imputed_matrix[:, j] = result_matrix[:, j] * allocated_total
    
    # Store the matrix in the dictionary
    sector_matrices[sector] = imputed_matrix

# Now, sector_matrices holds a 50x50 imputed service trade matrix for each sector (13-22)


/var/folders/g9/ggrmf_5j6tx4rv5qdhs0slq40000gn/T/ipykernel_45649/3098510060.py:25: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  allocated_total = grand_expenditure_shares[j] * sector_total


In [402]:
sector_matrices

{np.float64(14.0): array([[1.61044663e+03, 2.21676006e-02, 1.30936628e+01, ...,
         3.08129649e+00, 1.93301478e+01, 3.17735609e-01],
        [0.00000000e+00, 2.33476559e+02, 5.17244015e-02, ...,
         0.00000000e+00, 1.47784004e-02, 0.00000000e+00],
        [8.22417984e+00, 2.88178808e-01, 1.51018996e+03, ...,
         8.12812023e-02, 9.78330108e+00, 1.85468925e+00],
        ...,
        [1.19705043e+00, 7.38920021e-03, 1.09360163e+00, ...,
         3.34213526e+02, 3.46553490e+00, 1.03448803e-01],
        [2.32316455e+01, 5.46800816e-01, 2.28547963e+01, ...,
         5.71924096e+00, 2.04646856e+03, 1.26355324e+00],
        [5.91136017e-02, 7.38920021e-03, 1.07882323e+00, ...,
         7.38920021e-03, 6.57638819e-01, 1.54338225e+02]], shape=(50, 50)),
 np.float64(15.0): array([[4.34792782e+03, 5.98486940e-02, 3.53506286e+01, ...,
         8.31896847e+00, 5.21880612e+01, 8.57831281e-01],
        [0.00000000e+00, 6.30346395e+02, 1.39646953e-01, ...,
         0.00000000e+00, 3.9899

In [403]:
grand_expenditure_shares.values

array([0.01481573, 0.00168444, 0.013864  , 0.00751321, 0.11339449,
       0.01260871, 0.01059829, 0.00232349, 0.048189  , 0.03333345,
       0.00229637, 0.00373594, 0.05199033, 0.02704284, 0.01162791,
       0.01154655, 0.01591144, 0.02062573, 0.00304284, 0.01191783,
       0.01757134, 0.0357388 , 0.0182279 , 0.00875248, 0.01681641,
       0.00322738, 0.00692626, 0.00575783, 0.00398448, 0.0282656 ,
       0.00317333, 0.05069118, 0.02599761, 0.00371927, 0.04097326,
       0.01079054, 0.01046826, 0.0366341 , 0.00147691, 0.01437062,
       0.00261971, 0.02378887, 0.13206325, 0.00780187, 0.00139057,
       0.02008048, 0.02457121, 0.00393014, 0.02056139, 0.00156636])

In [404]:
import numpy as np

# Set tolerance for numerical comparisons
tol = 1e-8

# Loop through each imputed sector matrix (for sectors 13-22)
for sector, imputed_matrix in sector_matrices.items():
    # Retrieve the sector total from domestic_expenditure dataframe
    sector_total = domestic_expenditure.loc[
        domestic_expenditure['ExportingIndustry'] == sector, 'Value'
    ].values[0]
    
    print(f"\nChecking Sector {sector}:")
    
    # --- Overall Sum Check ---
    total_sum = imputed_matrix.sum()
    if np.allclose(total_sum, sector_total, atol=tol):
        print("  Overall sum check passed.")
    else:
        print(f"  Overall sum check failed: Sum = {total_sum}, Expected = {sector_total}")
    
    # --- Column Sums Check ---
    column_sums = imputed_matrix.sum(axis=0)
    expected_column_sums = grand_expenditure_shares * sector_total
    if np.allclose(column_sums, expected_column_sums, atol=tol):
        print("  Column sums check passed.")
    else:
        # Report differences for each column if they don't match
        for j in range(len(column_sums)):
            if not np.allclose(column_sums[j], expected_column_sums[j], atol=tol):
                print(f"  Column {j} sum check failed: Sum = {column_sums[j]}, Expected = {expected_column_sums[j]}")
    
    # --- Row Ratios Check for Each Column ---
    for j in range(imputed_matrix.shape[1]):
        allocated_total = expected_column_sums[j]
        # Only check ratios if allocated_total is greater than tolerance
        if allocated_total > tol:
            ratios = imputed_matrix[:, j] / allocated_total
            if np.allclose(ratios, result_matrix[:, j], atol=tol):
                print(f"  Column {j} row ratios check passed.")
            else:
                print(f"  Column {j} row ratios check failed.")
        else:
            # If allocated_total is nearly zero, expect the entire column to be zeros
            if np.allclose(imputed_matrix[:, j], 0, atol=tol):
                print(f"  Column {j} is all zeros as expected (allocated total is zero).")
            else:
                print(f"  Column {j} expected to be zero but is not (check failed).")



Checking Sector 14.0:
  Overall sum check passed.
  Column sums check passed.
  Column 0 row ratios check passed.
  Column 1 row ratios check passed.
  Column 2 row ratios check passed.
  Column 3 row ratios check passed.
  Column 4 row ratios check passed.
  Column 5 row ratios check passed.
  Column 6 row ratios check passed.
  Column 7 row ratios check passed.
  Column 8 row ratios check passed.
  Column 9 row ratios check passed.
  Column 10 row ratios check passed.
  Column 11 row ratios check passed.
  Column 12 row ratios check passed.
  Column 13 row ratios check passed.
  Column 14 row ratios check passed.
  Column 15 row ratios check passed.
  Column 16 row ratios check passed.
  Column 17 row ratios check passed.
  Column 18 row ratios check passed.
  Column 19 row ratios check passed.
  Column 20 row ratios check passed.
  Column 21 row ratios check passed.
  Column 22 row ratios check passed.
  Column 23 row ratios check passed.
  Column 24 row ratios check passed.
  Colu

/var/folders/g9/ggrmf_5j6tx4rv5qdhs0slq40000gn/T/ipykernel_45649/2893570910.py:35: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  allocated_total = expected_column_sums[j]


# sector_matrices is between-state, sectors 14-22

# Get rest of matrix. 

In [405]:
initial_emp = "/Users/jeffreyohl/Dropbox/SpatialW25/BigDataFiles/pset2/dummy data/L0_initial.csv"

In [406]:
L0_initial = pd.read_csv(initial_emp,header=None)

In [407]:
states = list(pivot_df.index.unique())

In [408]:
states

['Alabama',
 'Alaska',
 'Arizona',
 'Arkansas',
 'California',
 'Colorado',
 'Connecticut',
 'Delaware',
 'Florida',
 'Georgia',
 'Hawaii',
 'Idaho',
 'Illinois',
 'Indiana',
 'Iowa',
 'Kansas',
 'Kentucky',
 'Louisiana',
 'Maine',
 'Maryland',
 'Massachusetts',
 'Michigan',
 'Minnesota',
 'Mississippi',
 'Missouri',
 'Montana',
 'Nebraska',
 'Nevada',
 'New Hampshire',
 'New Jersey',
 'New Mexico',
 'New York',
 'North Carolina',
 'North Dakota',
 'Ohio',
 'Oklahoma',
 'Oregon',
 'Pennsylvania',
 'Rhode Island',
 'South Carolina',
 'South Dakota',
 'Tennessee',
 'Texas',
 'Utah',
 'Vermont',
 'Virginia',
 'Washington',
 'West Virginia',
 'Wisconsin',
 'Wyoming']

In [409]:
sectors = range(0,24)

In [410]:
L0_initial

,0
0,0.005356
1,0.000192
2,0.000356
3,0.000306
4,0.000017
...,...
1145,0.000019
1146,0.000169
1147,0.000155
1148,0.000074


# countries to states 

In [413]:
L_0_intial_vec = L0_initial[0]

In [414]:
states

['Alabama',
 'Alaska',
 'Arizona',
 'Arkansas',
 'California',
 'Colorado',
 'Connecticut',
 'Delaware',
 'Florida',
 'Georgia',
 'Hawaii',
 'Idaho',
 'Illinois',
 'Indiana',
 'Iowa',
 'Kansas',
 'Kentucky',
 'Louisiana',
 'Maine',
 'Maryland',
 'Massachusetts',
 'Michigan',
 'Minnesota',
 'Mississippi',
 'Missouri',
 'Montana',
 'Nebraska',
 'Nevada',
 'New Hampshire',
 'New Jersey',
 'New Mexico',
 'New York',
 'North Carolina',
 'North Dakota',
 'Ohio',
 'Oklahoma',
 'Oregon',
 'Pennsylvania',
 'Rhode Island',
 'South Carolina',
 'South Dakota',
 'Tennessee',
 'Texas',
 'Utah',
 'Vermont',
 'Virginia',
 'Washington',
 'West Virginia',
 'Wisconsin',
 'Wyoming']

In [415]:
L_0_intial_vec

0       0.005356
1       0.000192
2       0.000356
3       0.000306
4       0.000017
          ...   
1145    0.000019
1146    0.000169
1147    0.000155
1148    0.000074
1149    0.000188
Name: 0, Length: 1150, dtype: float64

In [416]:
import pandas as pd
import numpy as np

# Example setup:
# states = ["State1", "State2", ..., "State50"]  # length 50
# L0_initial = pd.Series(...)  # length 1150 (50 states × 23 sectors)

# 1) Create a DataFrame with columns: state, sector, share
df = pd.DataFrame({
    "state":  np.repeat(states, 23),
    "sector": np.tile(range(23), len(states)),
    "share":  L_0_intial_vec.values
})

# 2) Filter for sectors 1–22 only
df_filtered = df[df["sector"].between(1, 22)].copy()

# 3) For each sector, compute each state's fraction of total national employment
df_filtered["national_share"] = df_filtered.groupby("sector")["share"].transform(lambda x: x / x.sum())

wide_emp_shares = df_filtered


In [417]:
df_agg_sector_fix.columns

Index(['ExportingCountry', 'ExportingIndustry', 'ImportingCountry',
       'ImportingIndustry', 'Value'],
      dtype='object')

In [418]:
# For each sector (1 - 22) in df_agg_sector_fix

    # For each non-US country, take the trade flow (Into the US) in that sector with the US.
        # multiply by employment share of that state, in that sector [i.e. scale it down using national_share]

# So the goal will be , for each sector, a 37 x 50 matrix (where the 37 is the number of non-US countries)

# Row is origin country, column is destination (state)


In [419]:
import pandas as pd
import numpy as np

# --- Step 1: Set up the sorted lists for states and non-US countries ---

# Get all U.S. states from the employment shares (alphabetically sorted)
states = sorted(wide_emp_shares['state'].unique())

# Get the non-US exporting countries from the trade flow data,
# preserving alphabetical order.
countries = sorted(
    df_agg_sector_fix.loc[
        df_agg_sector_fix['ExportingCountry'] != us_code, 
        'ExportingCountry'
    ].unique()
)

# --- Step 2: Loop over each sector and allocate flows based on employment shares ---
# We'll store the result for each sector in a dictionary.

country_to_state_sector_matrices = {}

# Here we assume the sector values are stored in the 'ExportingIndustry' column.
# You could also use the 'ImportingIndustry' column since they are identical.
for sector in sorted(df_agg_sector_fix['ExportingIndustry'].unique()):
    # Filter for the flows in the current sector, where the US is the importer
    # and the exporting country is not the US.
    df_sector = df_agg_sector_fix[
        (df_agg_sector_fix['ExportingIndustry'] == sector) &
        (df_agg_sector_fix['ImportingCountry'] == us_code) &
        (df_agg_sector_fix['ExportingCountry'] != us_code)
    ]
    
    # In case there are multiple rows per country, sum the flows for each country.
    trade_by_country = df_sector.groupby('ExportingCountry')['Value'].sum()
    
    # Get employment shares for the given sector.
    # We re-index to include all states; missing states get a share of 0.
    emp_shares_sector = wide_emp_shares[
        wide_emp_shares['sector'] == sector
    ].set_index('state')['national_share'].reindex(states, fill_value=0)
    
    # Now we want an outer product: for each exporting country (row) multiply
    # its trade flow value by each state's employment share.
    # For countries that might not appear in the trade_by_country series, we assume a flow of 0.
    trade_vector = pd.Series(
        [trade_by_country.get(country, 0.0) for country in countries], 
        index=countries
    )
    
    # Compute the allocated flows: each cell is trade_value (for the country) * state share.
    # The result will have rows = countries and columns = states.
    allocated = pd.DataFrame(
        np.outer(trade_vector, emp_shares_sector),
        index=countries,
        columns=states
    )
    
    # Save the allocated matrix for this sector.
    country_to_state_sector_matrices[sector] = allocated

# --- sector_matrices now contains, for each sector (key), a DataFrame of shape
#      (number_of_non_US_countries, number_of_states) with the allocated flows.
#
# For example, to view the allocation for sector 2:
print("Sector 2 allocation matrix:")
print(country_to_state_sector_matrices[2])


Sector 2 allocation matrix:
       Alabama        Alaska       Arizona   Arkansas  California  \
1     2.797952  7.249897e-03  2.005241e-01   0.460677    7.819992   
2     1.627427  4.216896e-03  1.166346e-01   0.267952    4.548492   
3     3.732703  9.671971e-03  2.675159e-01   0.614582   10.432525   
4     0.062224  1.612306e-04  4.459458e-03   0.010245    0.173909   
5    22.361767  5.794256e-02  1.602626e+00   3.681819   62.498863   
6    67.150347  1.739962e-01  4.812541e+00  11.056165  187.678385   
7   148.911046  3.858499e-01  1.067218e+01  24.517894  416.191217   
8     0.000007  1.765214e-08  4.882384e-07   0.000001    0.000019   
9     1.498586  3.883051e-03  1.074009e-01   0.246739    4.188394   
10    0.433572  1.123446e-03  3.107327e-02   0.071387    1.211788   
11    0.036446  9.443578e-05  2.611988e-03   0.006001    0.101862   
12    0.753955  1.953605e-03  5.403452e-02   0.124137    2.107226   
13    7.451502  1.930791e-02  5.340353e-01   1.226874   20.826190   
14   1

# states to countries

In [420]:
# --- Step 2: Loop over each sector and allocate US export flows across states ---
# The resulting dictionary will map each sector to a 50 x 37 DataFrame where
# rows represent U.S. states and columns represent non‑US importing countries.
states_to_country_sector_matrices = {}

# We assume the sector indicator is stored in the 'ExportingIndustry' column.
for sector in sorted(df_agg_sector_fix['ExportingIndustry'].unique()):
    # Filter for the flows in the current sector where:
    # - The U.S. is the exporter.
    # - The destination (importing country) is not the U.S.
    df_sector = df_agg_sector_fix[
        (df_agg_sector_fix['ExportingIndustry'] == sector) &
        (df_agg_sector_fix['ExportingCountry'] == us_code) &
        (df_agg_sector_fix['ImportingCountry'] != us_code)
    ]
    
    # In case there are multiple rows per country, sum the export flows for each destination country.
    trade_by_country = df_sector.groupby('ImportingCountry')['Value'].sum()
    
    # Get employment shares for the given sector.
    # We re-index to include all states; missing states get a share of 0.
    emp_shares_sector = wide_emp_shares[
        wide_emp_shares['sector'] == sector
    ].set_index('state')['national_share'].reindex(states, fill_value=0)
    
    # Build a trade vector for the non-US importing countries.
    # For any country not present, we assume an export flow of 0.
    trade_vector = pd.Series(
        [trade_by_country.get(country, 0.0) for country in countries],
        index=countries
    )
    
    # Compute the allocated flows as the outer product of the state employment shares
    # and the export flows for each country.
    # This gives a matrix with rows = states and columns = non-US countries.
    allocated = pd.DataFrame(
        np.outer(emp_shares_sector, trade_vector),
        index=states,
        columns=countries
    )
    
    # Save the allocated matrix for this sector.
    states_to_country_sector_matrices[sector] = allocated

# --- Example: View the allocation for sector 2 ---
print("Sector 2 allocation matrix (rows: states, columns: non-US countries):")
print(states_to_country_sector_matrices[2])

Sector 2 allocation matrix (rows: states, columns: non-US countries):
                       1         2         3         4          5   \
Alabama          8.075171  0.373557  2.409301  0.037562   3.999616   
Alaska           0.020924  0.000968  0.006243  0.000097   0.010364   
Arizona          0.578733  0.026772  0.172670  0.002692   0.286645   
Arkansas         1.329560  0.061505  0.396686  0.006184   0.658528   
California      22.569282  1.044054  6.733750  0.104981  11.178520   
Colorado         0.616850  0.028535  0.184043  0.002869   0.305524   
Connecticut      1.006780  0.046574  0.300382  0.004683   0.498656   
Delaware         0.238274  0.011023  0.071091  0.001108   0.118016   
Florida          3.745705  0.173276  1.117565  0.017423   1.855240   
Georgia         15.410872  0.712906  4.597973  0.071684   7.632974   
Hawaii           0.286122  0.013236  0.085367  0.001331   0.141716   
Idaho            0.140953  0.006520  0.042054  0.000656   0.069814   
Illinois         2.3

In [421]:
thing = df_agg_sector_fix['ExportingCountry'].unique()

In [422]:
thing

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38])

In [482]:
df_agg_sector_USA_first = df_agg_sector_fix.copy()
df_agg_sector_USA_first['ImportingCountry'] = df_agg_sector_USA_first['ImportingCountry'].replace(38, 0)
df_agg_sector_USA_first['ExportingCountry'] = df_agg_sector_USA_first['ExportingCountry'].replace(38, 0)


In [483]:
df_agg_sector_USA_first

,ExportingCountry,ExportingIndustry,ImportingCountry,ImportingIndustry,Value
0,1,1.0,1,1.0,10851.759013
1,1,1.0,1,2.0,262.228559
2,1,1.0,1,3.0,40.510798
3,1,1.0,1,4.0,2.897519
4,1,1.0,1,5.0,352.067899
...,...,...,...,...,...
698891,0,22.0,0,18.0,214761.185588
698892,0,22.0,0,19.0,25490.798838
698893,0,22.0,0,20.0,241208.476296
698894,0,22.0,0,21.0,116928.130124


In [484]:
# Input costs
country_code = 0
destination_code = 5
source_code = 4


numerator = df_agg_sector_USA_first[(df_agg_sector_USA_first['ImportingCountry'] == country_code) &
                  (df_agg_sector_USA_first['ImportingIndustry'] == destination_code) & 
                  (df_agg_sector_USA_first['ExportingIndustry'] == source_code)].Value.sum()

# Revenues note the importing industry above is the exporting industry below
        #  note that importcountry above is expor|ting country below
denominator = df_agg_sector_USA_first[(df_agg_sector_USA_first['ExportingCountry'] == country_code) &
                  (df_agg_sector_USA_first['ExportingIndustry'] == destination_code)].Value.sum()

# each submatrix corresponds to a country
    # each row corresponds to a source sector
    # each column corresponds to an destination sector 

# values correspond to the quotient of numerator and denominator. 

In [ ]:
import numpy as np

# Suppose your DataFrame is named df_agg_sector_USA_first
# Columns: ['ExportingCountry', 'ExportingIndustry',
#           'ImportingCountry', 'ImportingIndustry', 'Value']

# 1) Identify unique countries and sectors
unique_countries = df_agg_sector_USA_first['ImportingCountry'].unique()
# or you might union the set of ExportingCountry and ImportingCountry
# if you need them all to match
unique_sectors = df_agg_sector_USA_first['ExportingIndustry'].unique()
# or union with ImportingIndustry if needed

unique_countries = np.sort(unique_countries)
unique_sectors = np.sort(unique_sectors)

# 2) Prepare a container for the ratio submatrices
#    For each country c, we want a matrix of shape (nSectors, nSectors)
#    submatrix[row=s, col=d].
input_output_submatrices = {}

# 3) Loop over each country and build the ratio matrix
for c in unique_countries:
    print(c)
    # Create an empty matrix for this country
    nSectors = len(unique_sectors)
    ratio_matrix = np.zeros((nSectors, nSectors))
    
    # 4) Compute the ratio for each (source_sector, destination_sector)
    for i, s in enumerate(unique_sectors):
        for j, d in enumerate(unique_sectors):
            
            # numerator: (ImportingCountry == c, ImportingIndustry == d, ExportingIndustry == s)
            numerator = df_agg_sector_USA_first[
                (df_agg_sector_USA_first['ImportingCountry'] == c) &
                (df_agg_sector_USA_first['ImportingIndustry'] == d) &
                (df_agg_sector_USA_first['ExportingIndustry'] == s)
            ]['Value'].sum()
            
            # denominator: (ExportingCountry == c, ExportingIndustry == d)
            denominator = df_agg_sector_USA_first[
                (df_agg_sector_USA_first['ExportingCountry'] == c) &
                (df_agg_sector_USA_first['ExportingIndustry'] == d)
            ]['Value'].sum()
            
            # Avoid division by zero
            ratio =      numerator / denominator if denominator != 0 else 0.0
            ratio_matrix[i, j] = ratio
    
    # Store this ratio matrix for country c
    input_output_submatrices[c] = ratio_matrix

# At the end, country_submatrices[c] holds a 2D


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37


In [510]:
sorted_countries = np.sort(unique_countries)

# 2) Retrieve each country’s ratio matrix from the dictionary
#    (assuming each ratio matrix is shape (nSourceSectors, nDestSectors))
matrices_in_order = [input_output_submatrices[c] for c in sorted_countries]

# 3) Vertically stack them into a single array
#    Result shape: ( (#countries * nSourceSectors), nDestSectors )
big_matrix = np.vstack(matrices_in_order)

df = pd.DataFrame(big_matrix)

# Save to CSV
df.to_csv( out_folder_IO+ "IO.csv",
          index=False,  # Set to True if you want row indices
          header=False)

In [512]:
big_matrix

array([[4.62390654e-01, 2.85271091e-02, 6.88051865e-03, ...,
        3.93990816e-01, 4.44552664e-01, 1.64997995e-02],
       [6.15135192e-04, 2.86312727e-01, 2.09474481e-02, ...,
        5.48224537e-02, 5.96116520e-03, 3.49672403e-03],
       [6.19145224e-02, 3.19497952e-02, 2.40186961e-01, ...,
        2.19464772e-01, 5.78288770e-02, 1.79785397e-02],
       ...,
       [5.42560966e-04, 4.79721883e-04, 5.05550833e-04, ...,
        4.25412669e-01, 5.38591452e-04, 2.61681585e-03],
       [8.59627092e-03, 4.95312316e-03, 3.77954331e-03, ...,
        2.99342702e-01, 1.58570101e-02, 5.14384219e-02],
       [8.58367803e-02, 4.33111462e-02, 2.54746102e-02, ...,
        1.03729822e+00, 7.49545270e-02, 2.28854463e-01]], shape=(836, 22))

In [ ]:
big_matrix

In [511]:
df

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,0.462391,0.028527,0.006881,0.001561,0.023108,0.012454,0.002789,0.000991,0.002827,0.001233,...,0.007573,0.002816,0.000706,0.000604,0.000155,0.000345,0.250880,0.393991,0.444553,0.016500
1,0.000615,0.286313,0.020947,0.000375,0.000659,0.012655,0.003106,0.000203,0.008059,0.000341,...,0.007689,0.006099,0.000353,0.002166,0.000130,0.000156,0.002147,0.054822,0.005961,0.003497
2,0.061915,0.031950,0.240187,0.000848,0.012462,0.037490,0.017792,0.011917,0.026871,0.010877,...,0.025331,0.132194,0.007789,0.018176,0.007218,0.005854,0.033169,0.219465,0.057829,0.017979
3,0.011435,0.013677,0.022652,0.053036,0.049955,0.024673,0.016720,0.009733,0.019214,0.007202,...,0.009417,0.209156,0.209483,0.002151,0.001748,0.003342,0.031574,0.201053,0.021839,0.049078
4,0.025833,0.177195,0.062952,0.025617,0.398074,0.317573,0.049218,0.022449,0.046191,0.026985,...,0.009164,0.046031,0.004313,0.007365,0.001591,0.007411,0.026177,0.670713,0.017752,0.020207
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
831,0.002532,0.001681,0.001256,0.000239,0.000415,0.001589,0.000765,0.000639,0.002048,0.001056,...,0.066523,0.004292,0.005923,0.021035,0.034546,0.064989,0.156848,0.218547,0.034304,0.029173
832,0.001284,0.000543,0.000511,0.000181,0.000674,0.000439,0.000421,0.000323,0.001270,0.000560,...,0.002716,0.003076,0.003267,0.003864,0.007373,0.001544,0.377160,0.150961,0.001386,0.011239
833,0.000543,0.000480,0.000506,0.000546,0.000296,0.000176,0.000335,0.000394,0.000942,0.000418,...,0.000844,0.001559,0.000868,0.001475,0.000656,0.000707,0.012466,0.425413,0.000539,0.002617
834,0.008596,0.004953,0.003780,0.000686,0.005605,0.003771,0.004070,0.002396,0.007981,0.004925,...,0.010660,0.018221,0.026452,0.007338,0.027141,0.007889,0.293505,0.299343,0.015857,0.051438


In [471]:
df_agg_sector_fix.ImportingCountry.unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38])

In [451]:
numerator/denominator

/var/folders/g9/ggrmf_5j6tx4rv5qdhs0slq40000gn/T/ipykernel_45649/2158541987.py:1: RuntimeWarning: invalid value encountered in scalar divide
  numerator/denominator


np.float64(nan)

# shitty value-added calculation, although this should be close to correct for the 
# 37 non-US countries?
# gamma_nj's

In [ ]:
import numpy as np

# Suppose 'big_matrix' is your original 836x22 array.
# 1) Reshape into 38 submatrices of shape 22x22
big_matrix_reshaped = big_matrix.reshape(38, 22, 22)

# 2) Sum each submatrix along the "row" axis => column sums
#    shape of col_sums will be (38, 22), i.e. one 22-element sum-vector per submatrix
one_minus_col_sums = 1-big_matrix_reshaped.sum(axis=1)

# 3) Construct the final 22x87 matrix
final_mat = np.zeros((22, 87))

# -- (a) First 50 columns all come from the first submatrix’s column sums
#       Repeat col_sums[0] (shape (22,)) across 50 columns
first_submatrix_sums = one_minus_col_sums[0]  # shape (22,)
final_mat[:, :50] = np.repeat(first_submatrix_sums[:, None], 50, axis=1)

# -- (b) Next 37 columns come from submatrices 2..38 (i.e. col_sums[1..37] in 0-based index)
for i in range(37):
    final_mat[:, 50 + i] = one_minus_col_sums[i + 1]

# final_mat is now the 22x87 matrix you described


# Gluing    

In [423]:
len(sector_matrices)

9

In [424]:
len(scaled_trade_flows)

13

In [425]:
df_agg_sector_fix.columns

Index(['ExportingCountry', 'ExportingIndustry', 'ImportingCountry',
       'ImportingIndustry', 'Value'],
      dtype='object')

In [426]:
# matrix-ise df_agg_sector_fix

In [427]:
import pandas as pd
import numpy as np

# 1. Filter out rows where ExportingCountry=38 or ImportingCountry=38
df_filtered = df_agg_sector_fix.query(
    "ExportingCountry != 38 & ImportingCountry != 38"
)

intl_sector_matrices = {}

# 2. Loop over each sector
for sector in range(1, 23):
    # 2a. Subset rows for this ExportingIndustry
    df_sector = df_filtered[df_filtered["ExportingIndustry"] == sector]
    
    # 2b. Pivot with aggfunc='sum' so that each (ExportingCountry, ImportingCountry)
    #     collapses into a single Value
    pivoted = df_sector.pivot_table(
        index="ExportingCountry",
        columns="ImportingCountry",
        values="Value",
        aggfunc="sum",
        fill_value=0
    )
    
    # 2c. Reindex to ensure the table is 1..37 × 1..37
    pivoted = pivoted.reindex(index=range(1, 38),
                              columns=range(1, 38),
                              fill_value=0)
    
    # 2d. Convert to NumPy array
    mat = pivoted.to_numpy()
    
    intl_sector_matrices[sector] = mat


# all of the belows are dicts of matrices. where the key is the sector

# Country-Country Trade
#### intl_sector_matrices    -  37x37 non-US trade for each of 22 sectors. sorted

# State-State Trade
#### scaled_trade_flows       -  50x50 for non-service sectors, which we have CFS data on (first 13)
#### sector_matrices - 50x50 for services sectors (14-22) 

# State-Country and Country-State trade
#### country_to_state_sector_matrices - 37 x 50 of countries to states for all 22 sectors, sorted.
#### states_to_country_sector_matrices  50 x 37 of state to country for all 22 sectors, sorted.

In [434]:
sector_keys = range(1,23)

In [435]:
import numpy as np

# Assuming sector_keys is a list of the sector keys in the correct order
final_matrix_list = []

for i, key in enumerate(sector_keys):
    # Select the appropriate state-to-state matrix based on the sector type
    if i < 13:  # sectors 1 to 13: non-services
        state_state = scaled_trade_flows[key]  # shape (50,50)
    else:       # sectors 14 to 22: services
        state_state = sector_matrices[key]     # shape (50,50)
    
    # Retrieve the state-to-country, country-to-state, and country-to-country matrices
    state_country   = states_to_country_sector_matrices[key]   # shape (50,37)
    country_state   = country_to_state_sector_matrices[key]      # shape (37,50)
    country_country = intl_sector_matrices[key]                # shape (37,37)
    
    # Create the 87x87 matrix for the current sector
    sector_matrix = np.block([
        [state_state,   state_country],
        [country_state, country_country]
    ])
    
    final_matrix_list.append(sector_matrix)

# Vertically stack all 22 sector matrices to form the final 1914x87 matrix
final_matrix = np.vstack(final_matrix_list)


In [441]:
df = pd.DataFrame(final_matrix)

# Save to CSV
df.to_csv(out_folder + "X_in_j.csv",
          index=False,  # Set to True if you want row indices
          header=False)

In [ ]:
scaled_trade_flows

In [430]:
sector_matrices

{np.float64(14.0): array([[1.61044663e+03, 2.21676006e-02, 1.30936628e+01, ...,
         3.08129649e+00, 1.93301478e+01, 3.17735609e-01],
        [0.00000000e+00, 2.33476559e+02, 5.17244015e-02, ...,
         0.00000000e+00, 1.47784004e-02, 0.00000000e+00],
        [8.22417984e+00, 2.88178808e-01, 1.51018996e+03, ...,
         8.12812023e-02, 9.78330108e+00, 1.85468925e+00],
        ...,
        [1.19705043e+00, 7.38920021e-03, 1.09360163e+00, ...,
         3.34213526e+02, 3.46553490e+00, 1.03448803e-01],
        [2.32316455e+01, 5.46800816e-01, 2.28547963e+01, ...,
         5.71924096e+00, 2.04646856e+03, 1.26355324e+00],
        [5.91136017e-02, 7.38920021e-03, 1.07882323e+00, ...,
         7.38920021e-03, 6.57638819e-01, 1.54338225e+02]], shape=(50, 50)),
 np.float64(15.0): array([[4.34792782e+03, 5.98486940e-02, 3.53506286e+01, ...,
         8.31896847e+00, 5.21880612e+01, 8.57831281e-01],
        [0.00000000e+00, 6.30346395e+02, 1.39646953e-01, ...,
         0.00000000e+00, 3.9899

In [428]:
states_to_country_sector_matrices

{np.float64(1.0):                        1         2          3         4          5   \
 Alabama          5.306458  0.652718   4.896340  0.039092   7.315907   
 Alaska           0.760040  0.093488   0.701299  0.005599   1.047852   
 Arizona          1.833968  0.225586   1.692227  0.013511   2.528454   
 Arkansas         7.846197  0.965118   7.239791  0.057802  10.817394   
 California      28.644636  3.523420  26.430790  0.211021  39.491786   
 Colorado         3.819954  0.469872   3.524723  0.028141   5.266494   
 Connecticut      1.820741  0.223959   1.680022  0.013413   2.510219   
 Delaware         1.127228  0.138654   1.040108  0.008304   1.554087   
 Florida          8.031765  0.987943   7.411017  0.059169  11.073233   
 Georgia         10.936873  1.345285  10.091600  0.080570  15.078448   
 Hawaii           0.932975  0.114760   0.860868  0.006873   1.286274   
 Idaho            2.826368  0.347656   2.607927  0.020821   3.896657   
 Illinois        15.635917  1.923289  14.427470

In [429]:
country_to_state_sector_matrices

{np.float64(1.0):       Alabama     Alaska    Arizona    Arkansas  California   Colorado  \
 1    5.017736   0.718687   1.734182    7.419288   27.086094   3.612112   
 2    0.560379   0.080263   0.193673    0.828584    3.024968   0.403400   
 3    2.335270   0.334479   0.807094    3.452960   12.605951   1.681088   
 4    0.082526   0.011820   0.028522    0.122024    0.445483   0.059408   
 5   10.211646   1.462607   3.529252   15.099071   55.123192   7.351047   
 6   65.125783   9.327918  22.508156   96.295819  351.553607  46.882031   
 7   16.784079   2.403971   5.800754   24.817155   90.601654  12.082338   
 8    0.002637   0.000378   0.000911    0.003899    0.014236   0.001898   
 9    0.398857   0.057128   0.137849    0.589755    2.153058   0.287125   
 10   1.271445   0.182108   0.439425    1.879975    6.863352   0.915274   
 11   0.068058   0.009748   0.023522    0.100632    0.367383   0.048993   
 12   0.258014   0.036955   0.089172    0.381503    1.392778   0.185736   
 13   2.

# Gross Output      

In [505]:

# A is your 1914x87 matrix
# Reshape A into 22 blocks of 87x87
blocks = final_matrix.reshape(22, 87, 87)

# For each block (each submatrix), sum across the columns (axis=2)
# This gives you an array of shape (22, 87)
row_sums = np.sum(blocks, axis=2)

# Transpose to get a final matrix of shape (87, 22)
GO = row_sums.T


In [509]:
df = pd.DataFrame(GO)

# Save to CSV
df.to_csv(out_folder_GO + "GO.csv",
          index=False,  # Set to True if you want row indices
          header=False)